In [ ]:
!pip install torch transformers faiss-cpu tqdm requests openpyxl tiktoken
!pip install bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 112.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 86.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 96.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 68.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 63.8 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12


In [ ]:
import faiss
print(faiss.__version__)


1.10.0


In [ ]:
!htop
!nvidia-smi


/bin/bash: line 1: htop: command not found
Wed Apr  9 18:40:38 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             43W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+----

In [ ]:
import torch

# Check if CUDA is available and the GPU is recognized
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("CUDA Version:", torch.version.cuda)
    print("PyTorch Version:", torch.__version__)


CUDA Available: True
GPU Name: NVIDIA A100-SXM4-40GB
CUDA Version: 12.4
PyTorch Version: 2.6.0+cu124


In [ ]:
import bitsandbytes as bnb
print("bitsandbytes version:", bnb.__version__)


bitsandbytes version: 0.45.5


In [ ]:
import torch
print(torch.cuda.is_available())  # Should return True


True


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import pandas as pd

afdb = pd.read_excel("/content/drive/MyDrive/Embeddings/data/AFDB.xlsx", engine="openpyxl")[["Filename", "Project Summary", "Project Components", "Sector"]]
aiib = pd.read_excel("/content/drive/MyDrive/Embeddings/data/AIIB.xlsx", engine="openpyxl")[["Filename", "Project Summary", "Project Components", "Sector"]]
idb = pd.read_excel("/content/drive/MyDrive/Embeddings/data/IDB.xlsx", engine="openpyxl")[["Filename", "Project Summary", "Project Components", "Sector"]]
adb = pd.read_excel("/content/drive/MyDrive/Embeddings/data/ADB.xlsx", engine="openpyxl")[["Filename", "Project Summary", "Project Components", "Sector"]]
world_bank = pd.read_excel("/content/drive/MyDrive/Embeddings/data/WBG_Updated.xlsx", engine="openpyxl")[["Project ID", "Project Components", "Sector"]]

# Define the columns for processing (include "Project Summary" and "Project Components")
aiib_columns = ["Project Summary", "Project Components"]
afdb_columns = ["Project Summary", "Project Components"]
world_bank_columns = ["Project Components"]
idb_columns = ["Project Summary", "Project Components"]
adb_columns = ["Project Summary", "Project Components"]

In [ ]:

import logging
import numpy as np
import pandas as pd
from tqdm import tqdm
import faiss
import json
import torch
import gc  # Garbage collection
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig
import torch.nn.functional as F
import time  # Timing execution
from torch.amp import autocast  # Mixed precision optimization



# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# Model setup
MODEL_NAME = "Alibaba-NLP/gte-Qwen2-7B-instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"

# 4-bit quantization (Optimized for NVIDIA L4 with 24GB VRAM)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,  # BF16 for stability
    bnb_4bit_use_double_quant=True  # Reduces memory overhead
)

# Load model efficiently
model = AutoModel.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",  # Automatic allocation of layers across GPU/CPU
    trust_remote_code=True
)

model.eval()  # Inference mode
torch.cuda.empty_cache()  # Free up GPU memory

# Verify device placement
print("Model device mapping:", model.hf_device_map)

# Tokenizer setup
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

# Embedding & FAISS configurations
CHUNK_SIZE = 4000
OVERLAP = 200
BATCH_SIZE = 4  # Reduce if OOM errors occur
EMBEDDING_DIM = 3584

# FAISS index setup
n_clusters = 14
faiss_index = faiss.IndexIVFFlat(
    faiss.IndexFlatL2(EMBEDDING_DIM), EMBEDDING_DIM, n_clusters
)
faiss_index.nprobe = 5
metadata_store = []

# Function to extract last token representation
def last_token_pool(last_hidden_states: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    sequence_lengths = attention_mask.sum(dim=1) - 1
    batch_size = last_hidden_states.shape[0]
    return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths]

# Function to chunk text
def chunk_text(text: str, chunk_size: int, overlap: int) -> list:
    if not text.strip():
        return []

    tokens = tokenizer.encode(text)
    chunks = [tokenizer.decode(tokens[i:i + chunk_size]) for i in range(0, len(tokens), chunk_size - overlap)]
    return chunks

# Function to generate embeddings
def generate_embeddings(texts, sectors):
    if not texts:
        return []

    prefixed_texts = [
        f"Sector: {sector}. Given the summary and components of a development aid project, retrieve similar projects: {text}"
        for text, sector in zip(texts, sectors)
    ]

    batch_dict = tokenizer(prefixed_texts, max_length=4000, padding=True, truncation=True, return_tensors="pt")
    batch_dict = {key: value.to(device) for key, value in batch_dict.items()}

    with torch.no_grad():
        with autocast(device_type="cuda", dtype=torch.float16):
            outputs = model(**batch_dict)
            embeddings = last_token_pool(outputs.last_hidden_state, batch_dict["attention_mask"])
            embeddings = F.normalize(embeddings, p=2, dim=1).detach()

    del batch_dict, outputs
    torch.cuda.empty_cache()
    gc.collect()

    return embeddings.cpu().numpy()  # Move to CPU for FAISS

# Function to train FAISS index
def train_faiss_index(df, section_columns, num_samples=500, batch_size=4):
    logging.info("Training FAISS index...")
    training_texts, training_sectors = [], []

    for _, row in df.iterrows():
        for section in section_columns:
            if pd.notna(row[section]):
                training_texts.append(str(row[section]))
                training_sectors.append(row.get("Sector", "Other"))
            if len(training_texts) >= num_samples:
                break
        if len(training_texts) >= num_samples:
            break

    if training_texts:
        embeddings = np.vstack([
            generate_embeddings(training_texts[i:i + batch_size], training_sectors[i:i + batch_size])
            for i in range(0, len(training_texts), batch_size)
        ])

        if not faiss_index.is_trained:
            faiss_index.train(embeddings.astype(np.float32))

        logging.info("FAISS training completed!")
        del embeddings
        gc.collect()

# Function to process DataFrame
def process_dataframe(df, project_type, id_column, section_columns):
    global metadata_store
    logging.info(f"Processing {project_type}, total rows: {len(df)}")

    for idx, row in tqdm(df.iterrows(), total=len(df)):
        data_id = row[id_column]
        sector = row.get("Sector", "Other")
        for section in section_columns:
            if pd.notna(row[section]):
                text = str(row[section])
                chunks = chunk_text(text, CHUNK_SIZE, OVERLAP)
                for i in range(0, len(chunks), BATCH_SIZE):
                    process_batch(chunks[i:i + BATCH_SIZE], data_id, project_type, section, sector, idx, i)

def process_batch(batch, data_id, project_type, section, sector, idx, chunk_offset):
    embeddings = generate_embeddings(batch, [sector] * len(batch))
    for chunk_idx, (embedding, chunk) in enumerate(zip(embeddings, batch)):
        faiss_index.add(embedding.astype(np.float32).reshape(1, -1))
        metadata_store.append({
            "project_type": project_type,
            "data_id": data_id,
            "section": section,
            "sector": sector,
            "chunk_id": f"chunk_{idx}_{chunk_offset + chunk_idx}",
            "chunk_content": chunk
        })

# 🚀 Train FAISS before processing
# 🚀 Train FAISS before processing with correct columns
combined_df = pd.concat([afdb, aiib, idb, world_bank, adb])

# Train FAISS with all sections available (excluding World Bank's "Project Summary")
train_faiss_index(combined_df, ["Project Summary", "Project Components"], num_samples=500)

# Process datasets
process_dataframe(adb, "ADB", "Filename", adb_columns)
process_dataframe(aiib, "AIIB", "Filename", aiib_columns)
process_dataframe(afdb, "AFDB", "Filename", afdb_columns)
process_dataframe(idb, "IDB", "Filename", idb_columns)
process_dataframe(world_bank, "World Bank", "Project ID", world_bank_columns)  # World Bank only processes "Project Components"

# Define Google Drive paths
FAISS_PATH = "/content/drive/MyDrive/Embeddings/faiss_index_QWEN_V7.idx"
METADATA_PATH = "/content/drive/MyDrive/Embeddings/metadata_store_QWEN_V7.json"

# Save FAISS index
faiss.write_index(faiss_index, FAISS_PATH)

# Save metadata
with open(METADATA_PATH, "w") as f:
    json.dump(metadata_store, f, indent=4)

logging.info(f"Processing complete. FAISS index saved at {FAISS_PATH} and metadata at {METADATA_PATH}.")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/902 [00:00<?, ?B/s]

modeling_qwen.py:   0%|          | 0.00/65.2k [00:00<?, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/Alibaba-NLP/gte-Qwen2-7B-instruct:
- modeling_qwen.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/80.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/370 [00:00<?, ?B/s]

100%|██████████| 2799/2799 [37:23<00:00,  1.25it/s]


In [ ]:
!htop
!nvidia-smi


/bin/bash: line 1: htop: command not found
Mon Mar 24 22:38:13 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   49C    P8             17W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+----